## Разрешение пространств имён и правило LEGB (Local, Enclosing, Global, Built-in)

In [6]:
print(str)

str = "global"

def outer():
    str = "enclosing"

    def inner():
        str = "local"
        print(str)

    inner()
    print(str)

outer()
print(str)

import builtins
str = builtins.str

<class 'str'>
local
enclosing
global


## Оператор global и nonlocal

In [7]:
a = 0
b = 0

def outer():
    a = 1
    b = 1

    def inner():
        global a
        # Изменение глобальной переменной a
        a = 2
        # Изменение внешней переменной b
        nonlocal b
        b = 2

    inner()
    # Переменная a осталась прежняя, b - изменена
    print(f"a = {a} b = {b}")

outer()
# Переменная a изменена, b - осталась прежняя
print(f"a = {a} b = {b}")

a = 1 b = 2
a = 2 b = 0


## Замыкание
Функция, которая запоминает значения переменных из внешней области видимости, даже после завершения работы внешней функции

In [ ]:
# Пример замыкания для создания функции сложения с запомненным значением
def add_number(a):
    def add(b):
        return a + b

    return add

# Вызывается функция add_number, в которую передаётся значение 5 в переменную a, а затем возвращается внутренняя функция add
add_five = add_number(5)
# При вызове функции add_five (которая указывает на фукцию add) с аргументом b,
# то к этому аргументу будет прибавляться запомненное ранее значение 5
print(add_five(2))  # Выведет 7
print(add_five(10)) # Выведет 15

7
15


In [ ]:
# Пример замыкания для создания счётчика
def make_counter():
    count = 0  # <-- внешняя переменная

    def inner():
        nonlocal count
        count += 1
        return count

    return inner

# make_counter() уже отработала и переменной count в программе больше нет
counter = make_counter()

# inner() запомнила переменную count — она хранится в замыкании
print(counter())  # Выведет 1
print(counter())  # Выведет 2
print(counter())  # Выведет 3

# кортеж ячеек (cell), внутри которых Python хранит захваченные переменные
print(counter.__closure__)

1
2
3
(<cell at 0x1062b4370: int object at 0x1038ded50>,)


## Принятие аргументов в функцию

In [13]:
a = [1, 2, 3]

def some(arg):
    # Проверка, ссылается ли аргумент внутри функции на объект снаружи
    print(arg is a)

## Список в функцию передался по ссылке, а не по значению
some(a)

True


In [16]:
# Как неправильно:
def some(a=[]):
    # Когда функция создаётся, python вычисляет значение по умолчанию и кладёт в память
    # Далее эти же значения будут использоваться каждый раз при вызове этой же функции, а не создаваться новые
    a.append(1)
    return a

print(some())
print(some([]))
# На этом момент происходит добавление элементу к предыдущему значению списка (Неправильно!)
print(some())
print(some([]))

# Как правильно:
def some(a=None):
    if a is None:
        a = []
    a.append(1)
    return a

print(some())
print(some([]))
print(some())
print(some([]))

[1]
[1]
[1, 1]
[1]
[1]
[1]
[1]
[1]


## *args и  *kwargs

In [17]:
def some(*args, **kwargs):
    # *args - позиционные неименованные аргументы (кортеж)
    # **kwargs - именованные аргументы в виде ключ-значение (словарь)
    print(locals())

some(1, 2, 3, a = 1, b = 2)

{'args': (1, 2, 3), 'kwargs': {'a': 1, 'b': 2}}


## Использование lambda

In [20]:
a = {
    1: 3,
    2: 2,
    3: 1
}

# В качестве критерия сравнения элементов передаётся lambda с аргументов в виде ключа словаря и возвращающая значение по этому ключу
print(max(a, key=lambda x: a[x]))

1


## Встроенные функции высшего порядка: map и filter

In [ ]:
a = [ i for i in range(5) ]

# Функция map создаёт генератор
a_map = map(lambda x: x**2, a)
print(a_map)
# Формируем список
print(list(a_map))

# Функция filter создаёт генератор
a_filter = filter(lambda x: x % 2 == 0, a)
print(a_filter)
# Формируем список
print(list(a_filter))

[0, 1, 4, 9, 16]
[0, 2, 4]


## Декоратор

In [ ]:
def decorator(func):
    # Замыкание, принимающее функцию
    def wrapper(*args, **kwargs):
        # Функция обёртка пробрасывает аргументы в исходную функцию
        # Выполняет исходную функцию, а также действия до неё и после, возвращает результат
        print("BEFORE")
        result = func(*args, *kwargs)
        print("AFTER")
        return result

    return wrapper

@decorator
def say_hi(name):
    print(f"Hi, my name is {name}")
    return 1

say_hi('John')

BEFORE
Hi, my name is John
AFTER


## Декоратор с аргументами

In [ ]:
def repeat(n):
    # Дополнительный внешний слой для аргументов декоратора
    def decorator(func):
        # Замыкание, принимающее функцию
        def wrapper(*args, **kwargs):
            # Функция обёртка пробрасывает аргументы в исходную функцию
            print("BEFORE")
            result = []
            for _ in range(n):
                r = func(*args, *kwargs)
                result.append(r)
            print("AFTER")
            return result

        return wrapper
    return decorator

@repeat(3)
def say_hi(name):
    print(f"Hi, my name is {name}")
    return 1

say_hi("Alex")

BEFORE
Hi, my name is Alex
Hi, my name is Alex
Hi, my name is Alex
AFTER


[1, 1, 1]

## Итерируемый объект

In [177]:
# Объект предоставляет возможность поочерёдного прохода по своим элементам
a = [1, 2, 3]

# Имеет метод __iter__, с помощью которого может быть обращён в итератор
print(hasattr(a, '__iter__'))  # True
print(hasattr(a, '__getitem__'))  # True
print(hasattr(a, '__next__'))  # False

True
True
False


## Итератор

In [178]:
# Объект, который запоминает текущее положение и умеет выдавать элементы один за другим
a = [1, 2, 3]

i = iter(a)
# Имеет метод __next__, с помощью которого можно брать некий следующий элемент
print(hasattr(i, '__iter__'))  # True
print(hasattr(a, '__getitem__'))  # True
print(hasattr(i, '__next__'))  # True

print(next(i))
print(next(i))

True
True
True
1
2


## Класс итератора

In [ ]:
class Counter:
    current_value: int

    def __init__(self):
        self.current_value = 0

    def __iter__(self):
        return self

    def __next__(self):
        current_value = self.current_value
        self.current_value += 1
        return current_value

c = Counter()

i = iter(c)
print(next(i))
print(next(i))

class Counter:
    current_value: int
    max_value: int

    def __init__(self, max_value):
        self.max_value = max_value
        self.current_value = 1

    def __iter__(self):
        return self

    def __next__(self):
        if self.current_value <= self.max_value:
            result = self.current_value
            self.current_value += 1
            return result
        else:
            raise StopIteration

# Использование итератора
c = Counter(5)
for num in c:
	print(num)

0
1
1
2
3
4
5


## Генератор

In [124]:
def some():
    yield 1

# При вызове функции получаем объект генератора
obj = some()
print(obj)
# Из объекта генератора можно по требованию вызывать функцию и забирать следующее вычисленное значение
print(next(obj))
# Если объектов не осталось, то будет ошибка StopIteration
# print(next(obj))

def some():
    # При указании несколько yield генератор запоминает место, после какого yield он возвратил значение
    # и затем после следующего вызова продолжит с этого места
    yield 1
    yield 2
    yield 3

obj = some()
print(next(obj))
print(next(obj))
print(next(obj))

def new():
    # Сначала будут возвращены значения из генератора some, затем из следующих yield
    yield from some()
    yield 4
    yield 5

obj = new()
for o in obj:
    print(o)

<generator object some at 0x104b7cf20>
1
1
2
3
1
2
3
4
5


In [155]:
def generator_example():
    value = yield
    yield value * 2

gen = generator_example()
next(gen)
# отправляет значение внутрь генератора
print(gen.send(10))

def generator_with_exception():
    try:
        yield
    except ValueError:
        yield "Exception caught"

gen = generator_with_exception()
next(gen)
# отправляет ошибку внутрь генератора
print(gen.throw(ValueError))

def generator_to_close():
    try:
        yield 1
    except GeneratorExit:
        print("Generator is being closed")

gen = generator_to_close()
next(gen)
# заканчивает итератор, вызываю ошибку GeneratorExit
gen.close()

20
Exception caught
Generator is being closed


## Генератор comprehensions

In [122]:
gen = ( i for i in range(5) if i % 2 == 0 )
print(gen)
print(next(gen))

for n in gen:
    print(n)

<generator object <genexpr> at 0x104b7cdd0>
0
2
4


## Менеджер контекста

In [3]:
# Позволяет обратиться к объекту и работать с полученными данными внутри этого контекста до выхода из него
# Если возникает ошибка, она может быть безопасно обработана
with open('data.txt', 'r') as file:
    # файл будет автоматически закрыт после выхода из блока with
    content = file.read()

print(content)

apple
banana
cherry


## Класс для менеджера контекста

In [129]:
class Connection:
    def __enter__(self):
        print("Открываю соединение при создании объекта")
        self.connection = 1
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        print("Закрываю соединение при выходе или ошибке")
        self.connection = 0

with Connection() as obj:
    print("Внутри контекста:", obj.connection)

print("После выхода:", obj.connection)

Открываю соединение при создании объекта
Внутри контекста: 1
Закрываю соединение при выходе или ошибке
После выхода: 0


## Функция с декоратором для менеджера контекста

In [131]:
from contextlib import contextmanager

@contextmanager
def connection():
    print("Открываю соединение")
    conn = {"state": 1}
    try:
        yield conn
    finally:
        print("Закрываю соединение")
        conn["state"] = 0

with connection() as conn:
    print("Внутри контекста:", conn["state"])

print("После выхода:", conn["state"])

Открываю соединение
Внутри контекста: 1
Закрываю соединение
После выхода: 0
